In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from phase_2.utils.paths import PROCESSED_DIR


In [2]:
SYMBOLS = [
    "SPY", "QQQ", "IWM",
    "AAPL", "MSFT", "NVDA",
    "XLU", "XLV", "XLP",
    "XLF", "XLE", "XLI",
]

data = {}

for sym in SYMBOLS:
    df = pd.read_parquet(PROCESSED_DIR / f"{sym.lower()}_daily.parquet")
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").reset_index(drop=True)
    df["year"] = df["date"].dt.year
    data[sym] = df

years = sorted(data["SPY"]["year"].unique())
years[:5], years[-5:]


([np.int32(2010),
  np.int32(2011),
  np.int32(2012),
  np.int32(2013),
  np.int32(2014)],
 [np.int32(2022),
  np.int32(2023),
  np.int32(2024),
  np.int32(2025),
  np.int32(2026)])

In [5]:
# Phase 2 strategies
from phase_2.scripts.strategies.trend.trend_strategy_v1 import run_trend_strategy_v1
from phase_2.scripts.strategies.meanrev.meanrev_strategy_v1 import run_meanrev_strategy_v1

# Phase 2 meta allocator & regime features
from phase_2.scripts.strategies.meta.meta_allocator_v1 import (
    build_regime_features,
    build_meta_raw_returns_param_v1,
)

# Phase 3 gates
from phase_3.strategies.meta.strategy_gates_v1 import apply_strategy_gates
# Portfolio plumbing
from phase_2.scripts.strategies.meta.risk_targeting_v1 import apply_vol_targeting
from phase_2.scripts.strategies.portfolio.portfolio_constructor_v1 import (
    align_asset_returns,
    compute_inverse_vol_weights,
    build_portfolio_raw_returns,
)


In [6]:
HARD_PARAMS_TIGHT = {
    "trend_mom60_min": 0.0,
    "trend_mom20_min": -0.005,
    "meanrev_mom20_max": -0.025,
    "meanrev_dd60_max": -0.03,
    "meanrev_vol20_max": 0.40,
}

In [ ]:
portfolio_all = []
weights_all = []   # store portfolio weights per day
returns_all = []   # store per-asset meta returns per day

for test_year in years:
    asset_daily = {}

    for sym, df in data.items():
        train = df[df["year"] < test_year]
        test  = df[df["year"] == test_year]

        if len(train) < 500 or len(test) < 60:
            continue

        trend = run_trend_strategy_v1(test, train_df=train)
        meanr = run_meanrev_strategy_v1(test)
        regime = build_regime_features(test)

        trend, meanr = apply_strategy_gates(trend, meanr, regime)

        meta = build_meta_raw_returns_param_v1(trend, meanr, regime, HARD_PARAMS_TIGHT)
            
        meta["asset"] = sym
        meta["year"] = test_year

        asset_daily[sym] = meta[["date", "asset", "year", "meta_raw_ret"]]

    if len(asset_daily) != len(SYMBOLS):
        continue

    # 1) Align asset returns
    ret_wide = align_asset_returns(asset_daily)   # index=date, columns=assets

    # 2) Compute portfolio weights (inverse-vol)
    w_assets = compute_inverse_vol_weights(ret_wide, lookback=20, max_weight=0.70)
    # w_assets: same index, same columns

    # 3) Build raw portfolio returns
    port_raw = build_portfolio_raw_returns(ret_wide, w_assets)

    # 4) Apply portfolio-level vol targeting
    vt = apply_vol_targeting(
        port_raw,
        target_vol_annual=0.10,
        lookback=20,
        max_leverage=1.0,
    )

    # Store portfolio series
    portfolio_all.append(
        pd.DataFrame({
            "date": port_raw.index,
            "portfolio_ret": vt["meta_ret"].values,
            "portfolio_lev": vt["lev"].values,
            "year": test_year,
        })
    )

    # Store weights (wide) with year for diagnostics
    w_df = w_assets.copy()
    w_df["date"] = w_df.index
    w_df["year"] = test_year
    weights_all.append(w_df.reset_index(drop=True))

    # Store per-asset returns (wide) for diagnostics
    r_df = ret_wide.copy()
    r_df["date"] = r_df.index
    r_df["year"] = test_year
    returns_all.append(r_df.reset_index(drop=True))


ValueError: Found array with 0 sample(s) (shape=(0, 5)) while a minimum of 1 is required by LinearRegression.

In [ ]:
portfolio = (
    pd.concat(portfolio_all)
    .sort_values("date")
    .reset_index(drop=True)
)

weights_long = (
    pd.concat(weights_all)
    .sort_values(["date"])
    .reset_index(drop=True)
)

returns_long = (
    pd.concat(returns_all)
    .sort_values(["date"])
    .reset_index(drop=True)
)

portfolio.head(), weights_long.head(), returns_long.head()


In [ ]:
def realized_vol_annual(returns, window=60):
    # returns: Series of daily returns
    vol_daily = returns.rolling(window).std()
    vol_annual = vol_daily * np.sqrt(252)
    return vol_annual

portfolio["realized_vol_60d"] = realized_vol_annual(portfolio["portfolio_ret"], window=60)

portfolio[["date", "portfolio_ret", "portfolio_lev", "realized_vol_60d"]].head()


In [ ]:
TARGET_VOL = 0.10

plt.figure(figsize=(12,5))
plt.plot(portfolio["date"], portfolio["realized_vol_60d"], label="Realized Vol (60d, annualized)")
plt.axhline(TARGET_VOL, linestyle="--", label="Target Vol", alpha=0.7)
plt.title("Portfolio Realized Vol vs Target (10%)")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
valid = portfolio.dropna(subset=["realized_vol_60d"])

plt.figure(figsize=(6,5))
plt.scatter(valid["realized_vol_60d"], valid["portfolio_lev"], alpha=0.3)
plt.xlabel("Realized Vol (60d, annualized)")
plt.ylabel("Portfolio Leverage")
plt.title("Leverage vs Realized Vol")
plt.grid(True)
plt.show()


In [ ]:
# Drop non-asset columns & compute long-horizon correlation
asset_cols = [c for c in returns_long.columns if c in SYMBOLS]

corr = returns_long[asset_cols].corr()
corr


In [ ]:
plt.figure(figsize=(8,6))
plt.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(label="Correlation")
plt.xticks(range(len(asset_cols)), asset_cols, rotation=45)
plt.yticks(range(len(asset_cols)), asset_cols)
plt.title("Asset Return Correlation Matrix")
plt.tight_layout()
plt.show()


In [ ]:
def effective_number_of_bets(weights_row: pd.Series) -> float:
    # weights_row: per-asset weights at a single date
    w = weights_row.values.astype(float)
    return 1.0 / np.sum(w**2)

enb_series = []

for _, row in weights_long.iterrows():
    w_row = row[asset_cols]
    enb = effective_number_of_bets(w_row)
    enb_series.append(enb)

weights_long["ENB"] = enb_series

plt.figure(figsize=(12,4))
plt.plot(weights_long["date"], weights_long["ENB"])
plt.title("Effective Number of Bets Over Time")
plt.ylabel("ENB")
plt.grid(True)
plt.show()


In [ ]:
# Use last 252 days for a snapshot
snapshot = returns_long.dropna().iloc[-252:]

cov = snapshot[asset_cols].cov()  # daily covariance
w_last = weights_long.dropna().iloc[-1][asset_cols].values  # last weights

w_vec = w_last.reshape(-1, 1)
port_var = float(w_vec.T @ cov.values @ w_vec)  # portfolio variance

# marginal contributions: (Σw)_i
marginal = cov.values @ w_vec  # shape (n,1)

# risk contribution of asset i: w_i * marginal_i
rc = (w_vec * marginal).flatten()

rc_frac = rc / port_var

risk_contrib = pd.DataFrame({
    "asset": asset_cols,
    "weight": w_last,
    "risk_contribution_frac": rc_frac,
}).sort_values("risk_contribution_frac", ascending=False)

risk_contrib
